In [1]:
import pandas as pd

WHO_URL = "https://github.com/Priyankkoul/Life-Expectancy-WHO---Data-Analytics/blob/master/DATASET.csv?raw=true"

who = pd.read_csv(WHO_URL)

print("Shape:", who.shape)

# first_5_checks(who, "WHO")

Shape: (2938, 22)


In [3]:
#1. Load & Inspect
import pandas as pd

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

cols = [
    'age','workclass','fnlwgt','education','education_num',
    'marital_status','occupation','relationship','race','sex',
    'capital_gain','capital_loss','hours_per_week',
    'native_country','income'
]

df = pd.read_csv(
    url,
    header=None,
    names=cols,
    na_values=' ?',
    skipinitialspace=True
)

print(df.shape)
print(df.dtypes)
print(df.isnull().sum())
df.head()

(32561, 15)
age                int64
workclass         object
fnlwgt             int64
education         object
education_num      int64
marital_status    object
occupation        object
relationship      object
race              object
sex               object
capital_gain       int64
capital_loss       int64
hours_per_week     int64
native_country    object
income            object
dtype: object
age               0
workclass         0
fnlwgt            0
education         0
education_num     0
marital_status    0
occupation        0
relationship      0
race              0
sex               0
capital_gain      0
capital_loss      0
hours_per_week    0
native_country    0
income            0
dtype: int64


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [4]:
#2. Define Target
df['income'] = (df['income'] == '>50K').astype(int)

df.drop('fnlwgt', axis=1, inplace=True)

df['income'].value_counts()

income
0    24720
1     7841
Name: count, dtype: int64

In [5]:
#3. Split First
from sklearn.model_selection import train_test_split

X = df.drop('income', axis=1)
y = df['income']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [6]:
#4. Impute Missing Values
from sklearn.impute import SimpleImputer

num_cols = X_train.select_dtypes(include='number').columns
cat_cols = X_train.select_dtypes(include='object').columns

num_imp = SimpleImputer(strategy='median')
cat_imp = SimpleImputer(strategy='most_frequent')

X_train[num_cols] = num_imp.fit_transform(X_train[num_cols])
X_test[num_cols] = num_imp.transform(X_test[num_cols])

X_train[cat_cols] = cat_imp.fit_transform(X_train[cat_cols])
X_test[cat_cols] = cat_imp.transform(X_test[cat_cols])

In [7]:
#5. Encode
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

X_train, X_test = X_train.align(
    X_test,
    join='left',
    axis=1,
    fill_value=0
)

print(X_train.shape)
print(X_test.shape)

(26048, 99)
(6513, 99)


In [8]:
#6. Scale
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [9]:
#7. Baseline
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

baseline = DummyClassifier(strategy='most_frequent')

baseline.fit(X_train, y_train)

y_pred = baseline.predict(X_test)

acc = accuracy_score(y_test, y_pred)

print("Baseline Accuracy =", acc)

Baseline Accuracy = 0.7592507293106096
